# D6 - Exception Hierarchies & Error Design

## Objective
Demonstrate custom exception hierarchies, error handling design, layer-specific exception mapping, and explicit exception chaining using `raise ... from ...` to inspect underlying root causes (`__cause__`).

## Concepts Covered
- **Custom Exception Hierarchy**: Establishing `TaskManagementError` as the domain root exception.
- **Layer-Specific Exceptions**: Granular exception types (`DataValidationError`, `ConfigError`, `ProcessingError`).
- **Exception Chaining (`raise ... from ...`)**: Preserving causal chain context via `__cause__`.
- **Defensive Error Handling**: Using `try / except / else / finally` blocks avoiding dangerous bare `except:` clauses.

## Project Implementation
The exception hierarchy resides in `app/utils/exceptions.py`:
- `TaskManagementError` (base domain exception subclassing `Exception`).
  - `DataValidationError` (raised when task payloads fail schema validation).
  - `ConfigError` (raised when configuration parameters are invalid).
  - `ProcessingError` (raised during pipeline stage execution failures).

## Demonstration
Below, we inspect the exception inheritance hierarchy, trigger domain exceptions, and inspect exception chaining causes.

In [1]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.utils.exceptions import (
    TaskManagementError,
    DataValidationError,
    ConfigError,
    ProcessingError
)

# 1. Verify Inheritance Hierarchy
print("Verifying Exception Hierarchy:")
print(f" - DataValidationError is subclass of TaskManagementError? {issubclass(DataValidationError, TaskManagementError)}")
print(f" - ConfigError is subclass of TaskManagementError? {issubclass(ConfigError, TaskManagementError)}")
print(f" - ProcessingError is subclass of TaskManagementError? {issubclass(ProcessingError, TaskManagementError)}")

Verifying Exception Hierarchy:
 - DataValidationError is subclass of TaskManagementError? True
 - ConfigError is subclass of TaskManagementError? True
 - ProcessingError is subclass of TaskManagementError? True


In [2]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.utils.exceptions import DataValidationError
from app.services.pipeline_stages import TaskValidationStep

validator = TaskValidationStep(title_required=True)
invalid_payload = {"priority": "URGENT"}  # Missing required 'title'

try:
    validator.process(invalid_payload)
except DataValidationError as e:
    print(f"Caught DataValidationError: {e}")
    print(f"Error Type: {type(e).__name__}")

Task title validation failed: empty or invalid length


Caught DataValidationError: TaskValidationStep: task title is missing or invalid; title is required and must be between 1 and 100 characters.
Error Type: DataValidationError


In [3]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.utils.exceptions import ProcessingError

# Demonstrate Exception Chaining (raise ... from ...) and __cause__ Inspection
def process_task_stage_with_chaining(data: dict):
    try:
        # Trigger an underlying low-level ValueError
        val = int(data["invalid_numeric_field"])
    except ValueError as raw_error:
        # Chain into domain ProcessingError
        raise ProcessingError("Pipeline stage processing failed due to invalid numeric input") from raw_error

try:
    process_task_stage_with_chaining({"invalid_numeric_field": "not_a_number"})
except ProcessingError as chained_err:
    print(f"Caught Top-Level Exception: {chained_err}")
    print(f"Inspecting __cause__: {chained_err.__cause__}")
    print(f"Original Root Cause Type: {type(chained_err.__cause__).__name__}")

Caught Top-Level Exception: Pipeline stage processing failed due to invalid numeric input
Inspecting __cause__: invalid literal for int() with base 10: 'not_a_number'
Original Root Cause Type: ValueError


## Actual Output
The code cells demonstrate:
1. Inheritance validation confirming all custom exceptions derive from `TaskManagementError`.
2. Interception of domain-specific `DataValidationError`.
3. Inspection of `chained_err.__cause__` revealing the original `ValueError` preserved via `raise ... from ...`.

## Key Observations
- Inheriting from a common base exception (`TaskManagementError`) allows global exception handling middleware to catch all domain errors cleanly.
- Explicit exception chaining (`raise NewException() from original_error`) preserves full stack trace context for debugging.
- Avoiding bare `except:` prevents unintentionally capturing system signals like `KeyboardInterrupt` or `SystemExit`.

## Conclusion
A clear, structured exception hierarchy improves system error design, simplifies debugging, and enables robust API error responses.